In [ ]:
import sys; sys.path.append("../..")

import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from pipeline.utils.ranking_utils import (
    K_FOLDS, K_PLACEBO, SEED, cv_folds, load_frame, make_arms, mean_metrics, placebo_frame,
    run_arm
)

DATA_DIR = Path("../../datasets/findhr")
JUDGE = "qwen3-8b"     # edit to switch judge
JUDGE_DIR = DATA_DIR / "judges" / JUDGE

In [ ]:
print(f"judge: {JUDGE}")
MODEL_DIR = JUDGE_DIR / "model"

ARMS = make_arms()

In [ ]:
df = load_frame(JUDGE)

In [ ]:
FOLDS = list(cv_folds(df, k=K_FOLDS, seed=SEED))

print(
    f"{df['id_j'].nunique()} queries -> {K_FOLDS} folds: "
    f"{[len(p['test'][1]) for p in FOLDS]} test queries each"
)

In [ ]:
CACHE = MODEL_DIR / "runs.pkl"
CACHED = pickle.loads(CACHE.read_bytes()) if CACHE.exists() else None

results = dict(CACHED["results"]) if CACHED else {}
missing = [a for a in ARMS if a != "Placebo" and a not in results]
if missing:
    print(f"fitting {missing}, {len(results)} arm(s) reused from {CACHE}")
    results |= {arm: run_arm(ARMS[arm], FOLDS, SEED) for arm in missing}
else:
    print(f"reusing {CACHE} - training skipped")

In [ ]:
# The paired tests below are only meaningful if position i is the same query in every arm.
REAL_ARMS = [a for a in ARMS.keys() if a != "Placebo"]

ref = results[REAL_ARMS[0]]["test_ids"]
ref_has_rel = results[REAL_ARMS[0]]["test"]["has_rel"]  # depends on the relevance only
for a in REAL_ARMS:
    assert np.array_equal(results[a]["test_ids"], ref), f"{a} tested a different query order"
    assert np.array_equal(results[a]["test"]["has_rel"], ref_has_rel)

print(f"{int(ref_has_rel.sum())} of {len(ref_has_rel)} queries evaluable; "
      f"{len(ref_has_rel) - int(ref_has_rel.sum())} have no relevant candidate and are excluded "
      f"from every mean (they stay in the training folds).")

In [ ]:
# Placebo: K_PLACEBO replicates, each a full K_FOLDS run with its own grid search on its own
# shuffle of PLACEBO_SOURCE. `placebo_means` (one mean nDCG@10 per replicate) is descriptive:
# nothing is tested against it. results["Placebo"] stacks the replicates as one arm for the
# plots. Cached replicates are reused by seed: placebo_frame and cv_folds are deterministic.
kept = [r for r in CACHED["placebo_runs"] if r["seed"] < K_PLACEBO] if CACHED else []
have = {r["seed"] for r in kept}
placebo_runs = list(kept)

for s in range(K_PLACEBO):
    if s in have:
        continue
    run = run_arm(
        cols=ARMS["Placebo"],
        folds=list(cv_folds(placebo_frame(df, s), k=K_FOLDS, seed=SEED)),
        seed=SEED
    )
    assert np.array_equal(run["test_ids"], ref), "placebo replicate saw a different partition"
    placebo_runs.append(dict(seed=s, **run))

placebo_runs.sort(key=lambda r: r["seed"])  # 7_gain_evaluation indexes these positionally

if missing or len(placebo_runs) != len(kept) or not CACHE.exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    CACHE.write_bytes(pickle.dumps({"results": results, "placebo_runs": placebo_runs}))
    print(f"cached {len(results)} arms + {len(placebo_runs)} placebo replicates to {CACHE}")

In [ ]:
results["Placebo"] = dict(
    cols=ARMS["Placebo"],
    test={
        m: np.concatenate([r["test"][m] for r in placebo_runs]) for m in placebo_runs[0]["test"].keys()
    },
    test_ids=np.tile(ref, K_PLACEBO),  # the same query list repeated once per replicate
)
# Spread across replicates: one mean nDCG@10 per replicate, over the evaluable queries.
placebo_means = np.array([mean_metrics(r["test"])["ndcg@10"] for r in placebo_runs])
print(f"Placebo nDCG@10 over {K_PLACEBO} replicates: mean={placebo_means.mean():.4f} "
      f"sd={placebo_means.std(ddof=1):.4f} max={placebo_means.max():.4f}")

In [ ]:
# Per-query nDCG of every arm, one row per (arm, replicate, query); `replicate` is -1 for the
# real arms and 0..K_PLACEBO-1 for the placebo. The figures and tables read this file.
RESULTS_DIR = JUDGE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

frames = [pd.DataFrame({"arm": arm, "replicate": -1, "id_j": results[arm]["test_ids"],
                        "has_rel": results[arm]["test"]["has_rel"],
                        "ndcg@10": results[arm]["test"]["ndcg@10"]})
          for arm in REAL_ARMS]
frames += [pd.DataFrame({"arm": "Placebo", "replicate": r["seed"], "id_j": r["test_ids"],
                         "has_rel": r["test"]["has_rel"],
                         "ndcg@10": r["test"]["ndcg@10"]})
           for r in placebo_runs]

ndcg_per_query = pd.concat(frames, ignore_index=True)
assert len(ndcg_per_query) == (len(REAL_ARMS) + K_PLACEBO) * len(ref)
ndcg_per_query.to_csv(RESULTS_DIR / "ndcg_per_query.csv", index=False)
print(f"wrote results/ndcg_per_query.csv | {ndcg_per_query.shape}")